# TripAdvisor Reviews – Debug & Test Notebook

Step-by-step testing for `src/sites/tripadvisor_reviews.py`.

Covers:
1. **API call** – fetching reviews via TripAdvisor Content API (full review text)
2. Ollama classification – topic & sentiment detection
3. Manual review input – adding reviews that the API can't fetch

TripAdvisor blocks direct web scraping (DataDome CAPTCHA) and Google Maps only
shows truncated snippets of TripAdvisor reviews. The Content API is the only
reliable source for full review text.

**Requirements:** Ollama running locally with `qwen2.5:7b`, `TRIPADVISOR_API_KEY` set.

In [1]:
from pathlib import Path
import sys, os, json

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Project root:', ROOT)

Project root: /home/laurabquintas/projects/reputation-analyzer


In [ ]:
from src.sites import tripadvisor_reviews as tr
import requests

api_key = os.getenv('TRIPADVISOR_API_KEY')
location_id = tr.LOCATION_IDS.get(tr.ANANEA_HOTEL)
print(f'Hotel: {tr.ANANEA_HOTEL}')
print(f'Location ID: {location_id}')
print(f'API key: {"set" if api_key else "NOT SET – set TRIPADVISOR_API_KEY to fetch reviews"}')

Hotel: Ananea Castelo Suites Hotel
Location ID: 33299137
API key: set


## 1. API Call – Single Page, English

Requires `TRIPADVISOR_API_KEY`. Returns max 5 reviews per page.

In [4]:
# Fetch a single page of English reviews (requires API key)
if api_key:
    reviews_en, total_en = tr.ta_get_reviews_page(location_id, api_key, language='en', offset=0)
    print(f'English reviews returned: {len(reviews_en)}')
    print(f'Total English reviews available: {total_en}')
    print()
    for r in reviews_en:
        print(f"  [{r.get('id')}] {r.get('rating')}★ – {r.get('title')} ({r.get('published_date', '')[:10]})")
else:
    print('TRIPADVISOR_API_KEY not set – skip API call')

English reviews returned: 3
Total English reviews available: 0

  [1037605287] 5★ – Lovely quiet hotel (2025-11-03)
  [1033353426] 5★ – Fabulous Modern Hotel (2025-10-04)
  [1033043012] 5★ – Wonderful week with wonderful staff (2025-10-02)


## 2. API Call – All Languages with Pagination

In [4]:
# Fetch across all default languages with pagination (requires API key)
if api_key:
    all_reviews = tr.ta_get_reviews(location_id, api_key)
    print(f'Total unique reviews fetched: {len(all_reviews)}')
    print()
    for r in sorted(all_reviews, key=lambda x: x.get('published_date', ''), reverse=True):
        lang = r.get('_language', '?')
        country = tr._extract_country(r)
        print(f"  [{r.get('id')}] {r.get('rating')}★ lang={lang} country={country or '?'} – {r.get('title')} ({r.get('published_date', '')[:10]})")
else:
    all_reviews = []
    print('TRIPADVISOR_API_KEY not set – skip API call')

Total unique reviews fetched: 5

  [1037859495] 5★ lang=de country=Germany – Wunderschöne Luxus Oase zur perfekten Entspannung (2025-11-05)
  [1037605287] 5★ lang=en country=England – Lovely quiet hotel (2025-11-03)
  [1033353426] 5★ lang=en country=County Meath – Fabulous Modern Hotel (2025-10-04)
  [1033204276] 4★ lang=de country=Germany – Joa, es geht auch besser.... (2025-10-03)
  [1033043012] 5★ lang=en country=England – Wonderful week with wonderful staff (2025-10-02)


In [5]:
# Inspect raw API response for first review
if all_reviews:
    print(json.dumps(all_reviews[0], indent=2, ensure_ascii=False))
else:
    print('No API reviews to inspect')

{
  "id": 1037605287,
  "lang": "en",
  "location_id": 33299137,
  "published_date": "2025-11-03T11:15:52Z",
  "rating": 5,
  "helpful_votes": 0,
  "rating_image_url": "https://www.tripadvisor.com/img/cdsi/img2/ratings/traveler/s5.0-66827-5.svg",
  "url": "https://www.tripadvisor.com/ShowUserReviews-g189112-d33299137-r1037605287-Reviews-Castelo_Suites_Hotel_ananea-Albufeira_Faro_District_Algarve.html?m=66827#review1037605287",
  "text": "We travelled as a family of four with 2 children 9 and 11 and really enjoyed our time at this hotel.\n\nThe hotel is in a quiet area with a few restaurants and bars within walking distance. There are a couple of beaches nearby (Praia do Evaristo and Praia do Castelo) both with restaurants and well worth a visit. I would recommend hiring a car, there is ample free parking available.\n\nWe stayed in a family suite with twin beds in one room and a king in the other. Each had its own door but were accessed by a single door at the front which could be locke

## 3. Ollama Classification – Test

Uses API reviews normalised to a common format:
`{id, rating, text, title, author_name, published_date}`.

In [6]:
# Check Ollama availability
from src.classification import classify_review, is_ollama_available, warm_up_model
from datetime import datetime
ollama_ok = is_ollama_available()
print(f'Ollama available: {ollama_ok}')

if ollama_ok:
    resp = requests.get('http://localhost:11434/api/tags')
    models = [m['name'] for m in resp.json().get('models', [])]
    print(f'Models: {models}')
    print('Warming up model...')
    warm_up_model()
    print('Model ready.')

Ollama available: True
Models: ['qwen2.5:7b', 'mistral:7b-instruct-q3_K_S', 'mistral:7b']
Warming up model...
Model ready.


In [8]:
# Build review list from API results (common format for classification)
reviews_for_classification = []

if 'all_reviews' in dir() and all_reviews:
    for r in all_reviews:
        reviews_for_classification.append({
            'id': str(r.get('id', '')),
            'rating': r.get('rating'),
            'title': r.get('title', ''),
            'text': r.get('text', ''),
            'published_date': r.get('published_date', ''),
            'author_name': (r.get('user') or {}).get('username', ''),
            'country': tr._extract_country(r) or 'Unknown',
            'travel_date': r.get('travel_date', ''),
            'trip_type': r.get('trip_type', '') or 'Unknown',
            'subratings': r.get('subratings', {}),
            'helpful_votes': r.get('helpful_votes', 0),
        })
    print(f'Using {len(reviews_for_classification)} reviews from TripAdvisor API')
else:
    print('No reviews available – set TRIPADVISOR_API_KEY and run Section 2 first')

if reviews_for_classification:
    r0 = reviews_for_classification[0]
    print(f'Sample: {r0.get("author_name", "?")} from {r0.get("country", "?")} – {r0.get("text", "")[:80]}...')

Using 5 reviews from TripAdvisor API
Sample: PippaNel from England – We travelled as a family of four with 2 children 9 and 11 and really enjoyed our...


In [ ]:
# Classify a single review (pick the first with text)
test_review = next((r for r in reviews_for_classification if r.get('text')), None)
if test_review and ollama_ok:
    print(f"Review by: {test_review.get('author_name', 'Unknown')}")
    title = test_review.get('title', '')
    if title:
        print(f"Title: {title}")
    print(f"Text: {test_review['text'][:300]}...")
    print()
    topics = classify_review(test_review['text'])
    print(f'Classification result ({len(topics)} topics):')
    for t in topics:
        icon = '\U0001f7e2' if t['sentiment'] == 'positive' else '\U0001f534'
        print(f"  {icon} {t['topic']} = {t['sentiment']}")
else:
    print('No review with text found or Ollama not available')

In [ ]:
# Classify ALL fetched reviews, save to JSON (overwrites existing reviews by ID)


if ollama_ok and reviews_for_classification:
    json_path = str(ROOT / 'data' / 'tripadvisor_reviews.json')
    existing = tr.load_reviews(json_path)
    existing_by_id = {r['id']: r for r in existing}

    for r in reviews_for_classification:
        text = r.get('text', '')
        if not text:
            continue
        topics = classify_review(text)

        review_id = str(r.get('id', ''))
        pub_date = r.get('published_date', '')
        if len(pub_date) == 10:
            pub_date = f"{pub_date}T00:00:00Z"

        record = {
            'id': review_id,
            'hotel': tr.ANANEA_HOTEL,
            'location_id': location_id or '',
            'source': 'tripadvisor',
            'rating': r.get('rating'),
            'title': r.get('title', ''),
            'text': text,
            'published_date': pub_date,
            'author_name': r.get('author_name', ''),
            'country': r.get('country', '') or 'Unknown',
            'travel_date': r.get('travel_date', ''),
            'trip_type': r.get('trip_type', '') or 'Unknown',
            'subratings': r.get('subratings', {}),
            'helpful_votes': r.get('helpful_votes', 0),
            'scraped_date': datetime.now().strftime('%Y-%m-%d'),
            'topics': topics,
            'classified': True,
        }

        action = 'updated' if review_id in existing_by_id else 'added'
        existing_by_id[review_id] = record

        stars = '\u2605' * int(r.get('rating') or 0)
        topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in topics)
        print(f"{stars} [{action}] {r.get('title') or r.get('author_name', 'N/A')[:45]:45s} \u2192 {topic_str or '(none detected)'}")

    all_records = list(existing_by_id.values())
    tr.save_reviews(all_records, json_path)
    print(f'\nSaved {len(all_records)} reviews to {json_path}')
else:
    print('Ollama not available or no reviews \u2013 skip classification')

★★★★★ [updated] Lovely quiet hotel                            → employees(pos), commodities(neg), comfort(neg), cleaning(pos), meals(pos), quality_price(pos), commodities(pos), return(pos)
★★★★★ [updated] Fabulous Modern Hotel                         → employees(pos), commodities(pos), cleaning(pos), meals(pos), quality_price(neg)
★★★★★ [updated] Wonderful week with wonderful staff           → employees(pos), commodities(pos), meals(pos)
★★★★★ [updated] Wunderschöne Luxus Oase zur perfekten Entspannung → employees(pos), commodities(pos), comfort(pos), commodities(neg), meals(pos)
★★★★ [updated] Joa, es geht auch besser....                  → employees(pos), commodities(pos), cleaning(pos), comfort(pos), comfort(neg), meals(neg), quality_price(neg), employees(neg)

Saved 11 reviews to /home/laurabquintas/projects/reputation-analyzer/data/tripadvisor_reviews.json


In [ ]:
# Debug: see the raw Ollama response for a specific review
# Change the index to test different reviews
DEBUG_INDEX = 0

if ollama_ok and reviews_for_classification:
    review = reviews_for_classification[DEBUG_INDEX]
    text = review.get('text', '')
    print(f"Review by: {review.get('author_name', 'Unknown')}")
    title = review.get('title', '')
    if title:
        print(f"Title: {title}")
    print(f"Full text:\n{text}\n")
    print('--- Sending to Ollama ---')

    topics = classify_review(text)
    print(f'\nClassification result ({len(topics)} topics):')
    for t in topics:
        icon = '🟢' if t['sentiment'] == 'positive' else '🔴'
        detail = f" – {t['detail']}" if t.get('detail') else ""
        print(f"  {icon} {t['topic']} = {t['sentiment']}{detail}")
else:
    print('Ollama not available or no reviews')

## 4. Manual Review Input

Since the API only returns 5 reviews per page, you can manually add reviews below.
Fill in the fields and run the cell to add them to the JSON file.

In [ ]:
from datetime import datetime
import hashlib

# ====== FILL IN YOUR REVIEW HERE ======
manual_review = {
    'reviewer_name': 'John D.',                    # reviewer name (used for ID)
    'rating': 5,                                    # 1-5
    'title': 'Amazing stay',                        # review title
    'text': 'Paste the full review text here...',   # full review text
    'published_date': '2026-03-01',                 # YYYY-MM-DD
    'trip_type': 'Couples',                         # Family, Couples, Solo, Business, Friends
    'country': 'Unknown',                           # reviewer country (or Unknown)
}
# ========================================

# Generate unique ID from name + date + title
id_seed = f"{manual_review['reviewer_name']}_{manual_review['published_date']}_{manual_review['title']}"
review_id = 'manual_' + hashlib.sha256(id_seed.encode()).hexdigest()[:12]

# Format published_date to match API format
pub_date = manual_review['published_date']
if len(pub_date) == 10:  # YYYY-MM-DD
    pub_date = f"{pub_date}T00:00:00Z"

record = {
    'id': review_id,
    'hotel': tr.ANANEA_HOTEL,
    'location_id': location_id,
    'rating': manual_review['rating'],
    'title': manual_review['title'],
    'text': manual_review['text'],
    'published_date': pub_date,
    'author_name': manual_review['reviewer_name'],
    'country': manual_review.get('country', '') or 'Unknown',
    'travel_date': '',
    'trip_type': manual_review.get('trip_type', '') or 'Unknown',
    'subratings': {},
    'helpful_votes': 0,
    'scraped_date': datetime.now().strftime('%Y-%m-%d'),
    'topics': [],
    'classified': False,
    'source': 'manual',
}

print(f'Generated ID: {review_id}')
print(json.dumps(record, indent=2, ensure_ascii=False))

In [ ]:
# Classify the manual review (optional)
if ollama_ok and record['text'] and record['text'] != 'Paste the full review text here...':
    topics = classify_review(record['text'])
    record['topics'] = topics
    record['classified'] = True
    print(f'Classification ({len(topics)} topics):')
    for t in topics:
        icon = '🟢' if t['sentiment'] == 'positive' else '🔴'
        print(f"  {icon} {t['topic']} = {t['sentiment']}")
else:
    print('Ollama not available or no text – skipping classification')

In [ ]:
# Save the manual review to the JSON file
json_path = str(ROOT / 'data' / 'tripadvisor_reviews.json')
existing = tr.load_reviews(json_path)

# Check for duplicates
existing_ids = {r['id'] for r in existing}
if record['id'] in existing_ids:
    print(f'⚠️  Review {record["id"]} already exists – skipping')
else:
    existing.append(record)
    tr.save_reviews(existing, json_path)
    print(f'✅ Saved! Total reviews: {len(existing)}')

## 5. Batch: Add Multiple Manual Reviews

Paste multiple reviews at once.

In [7]:
import hashlib
# Add multiple reviews at once
batch_reviews = [
    {
        'reviewer_name': 'Melanie',
        'rating': 5,
        'title': 'Wunderschöne Luxus Oase zur perfekten Entspannung',
        'text': 'Dieses Hotel ist ein wunderschöner Ort, um eine entspannte und erholsame Zeit an der Algarve zu verbringen. Das moderne und sehr exklusive Hotel unter einer sehr herzlichen und aufmerksamen Hotelmanagerin ist wirklich aussergewöhnlich geschmackvoll eingerichtet und bis ins kleinste Detail liebevoll gestaltet, so dass man die Harmonie des Hauses ständig spürt und erlebt! Die Farben und Materialen der Zimmer und des Interieurs greifen die Farben der Algarve perfekt auf und verschmelzen mit der wunderschönen Natur. Die Zimmer sind mit High-Tech und sehr hochwertigen Design-Möbeln ausgestattet, welche einen sofort in einen luxuriösen Urlaubsmodus versetzen. Der Aussenbereich war noch in Arbeit, aber man konnte auch hier schon die außergewöhnlich geschmackvolle und harmonische Bepflanzung erkennen. Auch das Frühstück war so liebevoll und reichhaltig mit lauter gesunden und frischen Köstlichkeiten dargeboten, dass alle Wünsche erfüllt wurden.',
        'published_date': '2025-11',
        'trip_type': 'Couple',
        'country': 'Germany',
    },
    {
        'reviewer_name': 'PippaNel',
        'rating': 4,
        'title': 'Lovely quiet hotel',
        'text': 'We travelled as a family of four with 2 children 9 and 11 and really enjoyed our time at this hotel. The hotel is in a quiet area with a few restaurants and bars within walking distance. There are a couple of beaches nearby (Praia do Evaristo and Praia do Castelo) both with restaurants and well worth a visit. I would recommend hiring a car, there is ample free parking available. We stayed in a family suite with twin beds in one room and a king in the other. Each had its own door but were accessed by a single door at the front which could be locked and created a bit of additional storage space. The rooms were small but perfectly adequate. Ritual toiletries were provided, and the rooms were cleaned and towels changed daily. There is a small fridge but no bottle openers in the rooms. We stayed half board and the food and service was beyond our expectations. Everything was freshly cooked, and the restaurant area was spotless. In the evening\'s salad, soups, vegetables and fresh meat / fish were offered as well as fruit, cheeses and a variety of desserts. The staff were great, even bringing salt for the poached eggs at breakfast without being asked. One slight disappointment was that the main pool isn\'t heated and was really cold. There is a heated pool on the roof terrace, but you had to be 16+ to access this. The hotel seemed to attract couples rather than families however, we had a great time and would definitely return.',
        'published_date': '2025-11',
        'trip_type': 'Family',
        'country': 'United Kingdom',
    },
    {
        'reviewer_name': 'cosshar',
        'rating': 5,
        'title': 'Fabulous Modern Hotel',
        'text': 'Fabulous hotel with a very calming feel. When you walk into the lobby it smells like a Spa. Beautiful modern rooms with high quality toiletries and daily servicing. Breakfast had something for everyone and a vast selection of hot food, pastries and juices to mention a few. Swimming pools were cold but still lovely and pool area is spotless. I highly recommend the Traditional Club Sandwich from the pool side bar..delicious! Staff couldn\'t be nicer, and always had a smile for you. Only a few € into the old town via uber too so very well located. 100% would visit again',
        'published_date': '2025-10',
        'trip_type': 'Couple',
        'country': 'Ireland',
    },
    {
        'reviewer_name': 'P E',
        'rating': 3,
        'title': 'Joa, es geht auch besser....',
        'text': 'Wir waren ende September im Hotel. Bauarbeiten, die aber nicht sonderlich störten. Bis auf die erheblichen Arbeitssicherheitsmängel!!!! So wurde vor dem Haupteingang der Schriftzug des Hotels angebracht, mit einem Hubsteiger. Fangnetz, Fallschutz, Sicherheitsnetz für herabfallende Gegenstände; KEINE! Die Zimmer groß, geräumig, leider wenig Stauraum. TV sehr intelligent untergebracht, fantastisch!!! Dusche + Toilette; EINE Schiebetür für Beides! Wenn einer duscht und der andere hat einen Toilettengang-----> je nachdem, wer die Schiebetür für sich beansprucht.... einer ist "praktisch" IM Zimmer. Ausreichend Steckdosen und Lademöglichkeiten vorhanden. Beim Verlassen des Zimmers wird ein blickdichter wärmeabweisender Vorhang autom. an der Fensterfront zugezogen, Sehr sehr gut! Spart Energie, die Klima hat weniger zu leisten! Top! Poolbereiche super gepflegt. Es werden Getränke und Snacks am Poolbereich serviert. Sehr ruhiges Hotel, auch der Gäste wegen. Kein Schlagen der Türen, Keine lautstarken Unterhaltungen auf den Fluren, keine deutlich wahrnehmbaren Toilettenspülgeräusche. Wenn die Fenster Nachts verschlossen sind ist Ruhe. Sonst die üblichen Vogelstimmen. Die Gäste(80%) waren überwiegend ü50 und ohne Kinder. Ja, die Drehtüren bereiteten immer noch Schwierigkeiten. Es ging nur einer von zwei Aufzügen. Personal ist sehr freundlich und zuvorkommend. Sprachschwierigkeiten (Englisch, mein level= B2) Das Personal spricht oft kein gutes englisch. Portugiesisch/Spanisch spreche ich nicht. Beim Personal wird dann sehr oft das Handy gezückt und mit "goo...-translate" weitergemacht. Nicht schön. Auch wir waren zu früh im Hotel und konnten das Zimmer erst nach 4h beziehen. Nervige Wartezeit. Hier hätten wir uns gewünscht, die Möglichkeit, Badesachen aus dem Koffer nehmen zu können. So liefen wir sinn und planlos die Wartezeit umher.',
        'published_date': '2025-10',
        'trip_type': 'Couple',
        'country': 'Germany',
    },
    {
        'reviewer_name': 'Helen T',
        'rating': 5,
        'title': 'Wonderful week with wonderful staff',
        'text': 'We have just checked out after a wonderful week in this hotel. The rooms are lovely, the staff are amazing and friendly. For such a young team of people they have the most excellent customer service skills. The breakfast was lovely too. We would definitely come back to this lovely place. Thank you x',
        'published_date': '2025-10',
        'trip_type': 'Couple',
        'country': 'United Kingdom',
    },
    {
        'reviewer_name': 'Destination649813',
        'rating': 1,
        'title': 'Shocking service',
        'text': 'Went with friends for a birthday cocktails and live music. Expensive average singer and they said no to ordering us a taxi at the end when we wanted to leave! Shocking as a local I will not go back. Luckily one of the group had the uber app.',
        'published_date': '2025-09',
        'trip_type': 'Friends',
        'country': 'Portugal',
    },
    {
        'reviewer_name': 'Sharon M',
        'rating': 4,
        'title': 'Comfortable modern hotel',
        'text': 'Just returned from a comfortable week in this modern hotel. Only issue we had was on our arrival (which we were aware was too early to check in) we were met by two young male staff members who provided no welcome or information they made it feel like we were entering "Fawlty Towers". We left our bags at reception and retreated to the pool bar for a refreshment after a short time we were contacted by a lovely female \'guest service agent\' who welcomed us and checked us into our room providing hotel information. Nice new clean room serviced daily. Top tip take coat hangers',
        'published_date': '2025-09',
        'trip_type': 'Couple',
        'country': 'United Kingdom',
    },
    {
        'reviewer_name': 'Hans N',
        'rating': 5,
        'title': 'Prima',
        'text': 'Nieuw hotel. Fantastische ervaring. Diner is erg goed. Mooi zwembad. Kamers zijn top. Ruim opgezet Prima parkeer mogelijkheid. Op loopafstand diverse strandjes. Toppie !',
        'published_date': '2025-09',
        'trip_type': 'Couple',
        'country': 'Netherlands',
    },
    {
        'reviewer_name': 'Diogo R',
        'rating': 5,
        'title': 'Best hotel in Albufeira for a calm and relaxing trip',
        'text': 'The hotel is very modern and well decorated. It\'s the perfect place to spend some time relaxing by the pool. The views amazing, either from the rooms or the rooftop bar. The service was outstanding, every employee is friendly and carrying, always available. During breakfast and dinner you have a wide range of delicious food. Really recommend for a calm vacation by the beach.',
        'published_date': '2025-09',
        'trip_type': 'Friends',
        'country': 'Portugal',
    },
    {
        'reviewer_name': 'Aaron',
        'rating': 3,
        'title': 'Verbesserungsbedürftiges 4-Sterne Hotel an der Algarve',
        'text': 'Viel Verbesserungspotenzial! Positiv: - Schöner Lobby/Bar-Bereich mit hochwertigem Mobiliar - Schönes Zimmer-Design mit bequemen Bett, extravaganten TV-Spiegel, schicker Badnische und hochwertigen Materialien - netter Balkon mit recht bequemen und hochwertigen Stühlen - bequeme Pool-Liegen mit hochwertigen Liegenauflagen und Alu-Gestell - hervorragendes (!) Frühstück mit frischem (!) und reifen Obst und nicht gepanschten Säften Sowie sehr guter, frischer Nespresso-Kaffee (!) am Automaten und leckeren, frischen, vielfältigen Backwaren - guter Ausgangspunkt für Ausflüge mit dem Mietwagen (ohne Mietwagen nur bedingt empfehlenswert) - wunderschöne Bucht in toller Kulisse nur 1 km entfernt Negativ: - über einen Monat nach Öffnung finden (Stand 09/25) immer noch Bauarbeiten im Hotel statt (z.B. Tiefgarage nicht fertiggestellt) -> tagsüber Baulärm überall hörbar - die hässliche Außenanlage eines Luxushotels, die ich bisher gesehen habe -> Pflanzenwelt nur in Akzenten vorhanden und teilweise abgestorben; hier fehlt jegliches Konzept und Harmonie; keine Wegesrand-Außenbeleuchtung - Pools sind nicht beheizt und bereits bei 28 Grad Außentemperatur und Sonne SEHR kalt - es gibt keinen Fitnessbereich, lediglich eine Art lieblos gestalteter und sinnbefreiter Trimm-Dich-Pfad - Zimmerreinigung miserabel, lieblos und mit eklatanten Sauberkeitsdefiziten (scheinbar Zeitarbeitsfirma beauftragt) - das gesamte Personal kann nicht den eleganten, teils luxuriösen Charakter des Hotels widerspiegeln -> in vielen Momenten unprofessionell, unaufmerksam, unsicher und stets ohne ein Lächeln auf den Lippen (und das einen Monat nach Öffnung des Hotels) - viel zu wenig Stauraum für die Klamotten im Zimmer - TV ohne deutsche Sender und Option etwas anzuschließen - Safe liegt lose in Schublade - alle Fensterflächen sind verschmutzt (Zimmer, Balkon, Außenbereiche) - bei Vollbelegung (wie bei uns) des Hotels gibt es Frühstückszeiten (drei Uhrzeiten wählbar) - das Restaurant ist viel zu klein konzipiert (bei Vollbelegung) - keine wechselnde Frühstücksspeisen - die Essensauswahl beim Abendessen ist sehr überschaubar -> insbesondere verglichen mit der Qualität der Frühstücksspeisen ist das Abendessen qualitativ weniger überzeugend; zum Teil schwach gewürzt, lauwarm, wiederholend - die Rooftop-Bar ist nicht in Betrieb; drei Mal täglich verirrt sich ein Kellner auf dem Rooftop Fazit: Es hapert an allen Ecken und Enden im ananea Castelo Suites. Die Führung des Hotels erscheint unerfahren und ohne nötiges Geschick, das potenzialreiche Hotel zu einem guten 4-Sterne-Hotel zu verschaffen. Insgesamt ist das Preis-Leistungsverhältnis nicht gut, obwohl es nur wenig benötigt, um einen schönen und zufriedenstellenden Aufenthalt zu ermöglichen. „Alle" genannten Punkte wurden während des Aufenthalts bereits mit dem Management besprochen und man ist uns bereits Entgegengekommen. Daher vergeben wir gut gemeinte 3 von 5 Sternen.',
        'published_date': '2025-09',
        'trip_type': 'Couple',
        'country': 'Germany',
    },
    {
        'reviewer_name': 'Markus J. H.',
        'rating': 5,
        'title': 'Wirklich toll',
        'text': 'Ein wirklich wunderbares Hotel! Das Hotel wurde erst Ende Juli 2025 eröffnet und ist daher komplett neu. Die erst kurz zurückliegende Neueröffnung merkt man kaum, vielmehr haben sich die Abläufe schon sehr gut eingespielt. Allerdings haben derzeit noch nicht alle Bereiche des Hotels geöffnet, insbesondere ist der Spa-Bereich offenbar noch nicht fertig. Die Lage des Hotels ist gut: Es sind fußläufig in 15-20 Minuten mehrere sehr schöne Strandabschnitte der Algarve zu erreichen und es gibt auch ausreichend Restaurants in der Umgebung, wobei man sich bei der Auswahl vorher anhand von Rezensionen informieren sollte (es gibt da auch einige eher schlechte). Die Zimmer des Hotels sind sehr schön, hell, alle mit einem Balkon ausgestattet und haben - jedenfalls in den oberen Etagen - allesamt Meerblick. In den Zimmern gibt es noch ein paar kleinere „Kinderkrankheiten", wie eine noch im Wind scheppernde Balkonabtrennung oder ein leicht tropfender Siphon am Waschbecken - diese dürften aber sicher sehr zeitnah behoben sein. Pool ist sehr groß und verfügt über eine - jedenfalls derzeit - mehr als ausreichende Anzahl an Liegen. Sehr überzeugt hat uns das Frühstück, das wirklich keine Wünsche offen lässt. Noch mehr überzeugt hat uns das Abendessen, zu welchem für einen verhältnismäßig geringen Betrag von €30 ein reichhaltiges Buffet aus täglich wechselnden kulinarischen Regionen angeboten wird. Äußerst begeistert waren wir schließlich vom außerordentlich freundlichen und kompetenten Personal, das wirklich alles versucht, um dem Gast einen schönen Aufenthalt zu bereiten. Wir haben uns sehr wohl gefühlt und kommen gern wieder!',
        'published_date': '2025-09',
        'trip_type': 'Couple',
        'country': 'Germany',
    },
    {
        'reviewer_name': 'Relax18746311737',
        'rating': 5,
        'title': 'Um lugar para relaxar e aproveitar',
        'text': 'Adorei o hotel! A arquitetura é lindíssima e os espaços exteriores são incríveis. O cantinho da biblioteca é super acolhedor, perfeito para relaxar e ler sem pressa. Os cocktails estavam ótimos e o pôr do sol… simplesmente 10/10, daqueles que ficam na memória!',
        'published_date': '2025-08',
        'trip_type': 'Friends',
        'country': 'Portugal',
    },
]


json_path = str(ROOT / 'data' / 'tripadvisor_reviews.json')
existing = tr.load_reviews(json_path)
existing_ids = {r['id'] for r in existing}
added = 0

for mr in batch_reviews:
    id_seed = f"{mr['reviewer_name']}_{mr['published_date']}_{mr['title']}"
    rid = 'manual_' + hashlib.sha256(id_seed.encode()).hexdigest()[:12]
    
    if rid in existing_ids:
        print(f'⚠️  Skip duplicate: {mr["title"]}')
        continue
    
    pub_date = mr['published_date']
    if len(pub_date) == 10:
        pub_date = f"{pub_date}T00:00:00Z"
    
    rec = {
        'id': rid,
        'hotel': tr.ANANEA_HOTEL,
        'location_id': location_id,
        'rating': mr['rating'],
        'title': mr['title'],
        'text': mr['text'],
        'published_date': pub_date,
        'author_name': mr['reviewer_name'],
        'country': mr.get('country', '') or 'Unknown',
        'travel_date': '',
        'trip_type': mr.get('trip_type', '') or 'Unknown',
        'subratings': {},
        'helpful_votes': 0,
        'scraped_date': datetime.now().strftime('%Y-%m-%d'),
        'topics': [],
        'classified': False,
        'source': 'manual',
    }
    
    # Classify if Ollama available
    if ollama_ok and rec['text'] and 'Replace with' not in rec['text']:
        try:
            topics = classify_review(rec['text'])
            rec['topics'] = topics
            rec['classified'] = True
        except Exception as e:
            print(f'  Classification failed: {e}')
    
    existing.append(rec)
    existing_ids.add(rid)
    added += 1
    topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in rec.get('topics', []))
    print(f"✅ {mr['title']} → {topic_str or '(unclassified)'}")

if added:
    tr.save_reviews(existing, json_path)
    print(f'\nSaved {added} new reviews. Total: {len(existing)}')
else:
    print('No new reviews to add.')

✅ Wunderschöne Luxus Oase zur perfekten Entspannung → (unclassified)
✅ Lovely quiet hotel → employees(pos), meals(pos), commodities(neg), return(pos)
✅ Fabulous Modern Hotel → employees(pos), meals(pos), meals(neg), commodities(pos), cleaning(pos), return(pos)
✅ Joa, es geht auch besser.... → employees(neg), commodities(pos), commodities(neg), cleaning(pos), comfort(neg), employees(pos), quality_price(neg), cleaning(neg)
✅ Wonderful week with wonderful staff → employees(pos), meals(pos), return(pos)
✅ Shocking service → employees(neg), return(neg)
✅ Comfortable modern hotel → employees(neg), employees(pos), cleaning(pos)
✅ Prima → (unclassified)
✅ Best hotel in Albufeira for a calm and relaxing trip → (unclassified)
✅ Verbesserungsbedürftiges 4-Sterne Hotel an der Algarve → employees(neg), commodities(neg), cleaning(neg), comfort(neg), quality_price(neg), meals(neg), return(neg)
✅ Wirklich toll → (unclassified)
✅ Um lugar para relaxar e aproveitar → (unclassified)

Saved 12 new reviews

## 6. Reclassify Reviews with Empty Topics

Finds reviews that were classified but got 0 topics (LLM missed), and retries.

In [8]:
json_path = str(ROOT / 'data' / 'tripadvisor_reviews.json')
reviews = tr.load_reviews(json_path)

# Find reviews that are "classified" but have 0 topics – likely LLM failures
needs_retry = [r for r in reviews if r.get('classified') and not r.get('topics') and r.get('text')]
# Also find unclassified reviews
unclassified = [r for r in reviews if not r.get('classified') and r.get('text')]

print(f'Total reviews: {len(reviews)}')
print(f'Classified with 0 topics (needs retry): {len(needs_retry)}')
print(f'Unclassified: {len(unclassified)}')

to_reclassify = needs_retry + unclassified

if ollama_ok and to_reclassify:
    for r in to_reclassify:
        try:
            topics = classify_review(r['text'])
            r['topics'] = topics
            r['classified'] = True
            topic_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in topics)
            author = r.get('title') or r.get('author_name', 'N/A')
            print(f"  ✅ {author[:40]} → {topic_str or '(still none)'}")
        except Exception as e:
            author = r.get('title') or r.get('author_name', 'N/A')
            print(f"  ❌ {author[:40]} → Error: {e}")
    
    tr.save_reviews(reviews, json_path)
    print(f'\nDone. Saved {len(reviews)} reviews.')
elif not ollama_ok:
    print('Ollama not available')
else:
    print('Nothing to reclassify')

Total reviews: 12
Classified with 0 topics (needs retry): 5
Unclassified: 0
  ✅ Wunderschöne Luxus Oase zur perfekten En → (still none)
  ✅ Prima → (still none)
  ✅ Best hotel in Albufeira for a calm and r → (still none)
  ✅ Wirklich toll → (still none)
  ✅ Um lugar para relaxar e aproveitar → (still none)

Done. Saved 12 reviews.


## 7. Reclassify ALL Reviews

Re-runs classification on **every** review with text. Use this after changing
the classification prompt in `src/classification.py`.

In [ ]:
json_path = str(ROOT / 'data' / 'tripadvisor_reviews.json')
reviews = tr.load_reviews(json_path)

with_text = [r for r in reviews if r.get('text')]
print(f'Total reviews: {len(reviews)}')
print(f'Reviews with text (will reclassify): {len(with_text)}')

if ollama_ok and with_text:
    for i, r in enumerate(with_text, 1):
        try:
            old_topics = r.get('topics', [])
            new_topics = classify_review(r['text'])
            r['topics'] = new_topics
            r['classified'] = True

            old_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in old_topics)
            new_str = ', '.join(f"{t['topic']}({t['sentiment'][:3]})" for t in new_topics)
            changed = '🔄' if old_str != new_str else '✅'
            author = r.get('title') or r.get('author_name', 'N/A')
            print(f"  {changed} [{i}/{len(with_text)}] {author[:40]}")
            if old_str != new_str:
                print(f"       old: {old_str or '(none)'}")
                print(f"       new: {new_str or '(none)'}")
        except Exception as e:
            author = r.get('title') or r.get('author_name', 'N/A')
            print(f"  ❌ [{i}/{len(with_text)}] {author[:40]} → Error: {e}")

    tr.save_reviews(reviews, json_path)
    print(f'\nDone. Reclassified {len(with_text)} reviews. Saved {len(reviews)} total.')
elif not ollama_ok:
    print('Ollama not available')
else:
    print('No reviews with text found.')

## 8. Current Data Summary

In [ ]:
import pandas as pd

json_path = str(ROOT / 'data' / 'tripadvisor_reviews.json')
reviews = tr.load_reviews(json_path)

print(f'Total reviews: {len(reviews)}')
print(f'Classified: {sum(1 for r in reviews if r.get("classified"))}')
print(f'With topics: {sum(1 for r in reviews if r.get("topics"))}')
print(f'Manual: {sum(1 for r in reviews if r.get("source") == "manual")}')
print()

# Country breakdown
country_counts = {}
for r in reviews:
    c = r.get('country', 'Unknown') or 'Unknown'
    country_counts[c] = country_counts.get(c, 0) + 1
if country_counts:
    print('Country breakdown:')
    for c, n in sorted(country_counts.items(), key=lambda x: -x[1]):
        print(f'  {c}: {n}')
    print()

# Topic breakdown
topic_counts = {}
for r in reviews:
    for t in r.get('topics', []):
        key = f"{t['topic']} ({t['sentiment']})"
        topic_counts[key] = topic_counts.get(key, 0) + 1

if topic_counts:
    df = pd.DataFrame.from_dict(topic_counts, orient='index', columns=['Count'])
    df = df.sort_index()
    print('Topic sentiment counts:')
    print(df.to_string())
else:
    print('No topics classified yet.')